# Privora — Private Document Intelligence Workbench
### Google Colab Demonstration & Headless Engine Tour
**Private On-Premise Document Intelligence & Tabular Analytics**

---

> **Important Environment Context & Disclaimer**:
> - This notebook runs in **Google Colab's cloud Linux environment (Ubuntu)** and demonstrates the **headless Python backend**, document ingestion pipeline, BM25 retrieval, and REST API endpoints.
> - **Privora's full Windows Desktop Edition** (featuring Microsoft Edge App Mode, native launcher Privora.exe, and Inno Setup installer) runs natively on on-premise Windows workstations.
> - In high-security enterprise production, Privora is deployed **completely air-gapped** on local workstations with zero internet connectivity. This Colab demo provides an interactive cloud preview for evaluation purposes.

## Step 1: Clone Repository & Install Python Dependencies
We install the required document extraction libraries, BM25 Okapi retrieval engine, FastAPI server, and testing tools.

In [ ]:
# 1. Clone repository if not already present
import os
from pathlib import Path

repo_name = "privora"
if not Path("backend").exists():
    print("Cloning repository...")
    !git clone https://github.com/your-username/privora.git
    %cd privora
else:
    print("Privora repository structure detected in current directory.")

# 2. Install Python dependencies
!pip install -q fastapi uvicorn python-multipart pydantic pypdf pymupdf python-docx openpyxl rank-bm25 httpx requests rapidocr-onnxruntime pillow

## Step 2: (Optional) Local Ollama Installation in Colab
Privora uses **Ollama** running locally on `http://127.0.0.1:11434` for 100% on-premise neural answer generation.  
On Google Colab, we can install the Linux Ollama daemon and download a quantized reasoning model (such as `qwen2.5:1.5b` or `qwen2.5:7b`).

*Note: If Ollama is not installed or GPU memory is limited, Privora automatically operates in **Offline Evidence Mode**—document parsing, passage indexing, BM25 keyword retrieval, and deterministic tabular math remain 100% operational.*

In [ ]:
# Install Ollama binary in Colab Linux environment and start background service
import subprocess, time, urllib.request

def setup_colab_ollama():
    try:
        # Check if Ollama is already installed
        res = subprocess.run(["which", "ollama"], capture_output=True, text=True)
        if not res.stdout.strip():
            print("Installing Ollama on Colab...")
            !curl -fsSL https://ollama.com/install.sh | sh
        
        # Start Ollama service in background
        print("Starting Ollama daemon in background...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        
        # Pull lightweight Qwen 2.5 model suitable for Colab T4 GPU
        print("Pulling qwen2.5:1.5b model...")
        !ollama pull qwen2.5:1.5b
        print("Ollama setup complete!")
    except Exception as e:
        print(f"Ollama setup notice: {e}. Running in Offline Evidence Mode.")

# Uncomment the following line to run local Ollama on Colab (requires GPU runtime):
# setup_colab_ollama()

## Step 3: Initialize Privora Backend Engine
We set the `PYTHONPATH` to include the `backend/` directory and create a TestClient to interact directly with the FastAPI application.

In [ ]:
import sys
import os
from pathlib import Path

# Configure PYTHONPATH
backend_path = Path.cwd() / "backend"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# Health check
resp = client.get("/api/health")
print("Health Status:", resp.status_code)
print(resp.json())

## Step 4: Multi-Format Document Ingestion Demonstration
Privora natively supports **PDF, DOCX, ODT, XLSX, CSV, and TXT** documents without sending any data to external parsers.  
Below, we create an in-memory sample industrial **OpenDocument Text (.odt)** file and an **Excel Spreadsheet (.xlsx)**, upload them, and verify SQLite passage chunking.

In [ ]:
import io, zipfile
import openpyxl

# 1. Create a sample ODT document
odt_buf = io.BytesIO()
with zipfile.ZipFile(odt_buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("mimetype", "application/vnd.oasis.opendocument.text")
    content_xml = '''<?xml version="1.0" encoding="UTF-8"?>
<office:document-content xmlns:office="urn:oasis:names:tc:opendocument:xmlns:office:1.0"
                         xmlns:text="urn:oasis:names:tc:opendocument:xmlns:text:1.0"
                         xmlns:table="urn:oasis:names:tc:opendocument:xmlns:table:1.0">
  <office:body>
    <office:text>
      <text:h text:outline-level="1">Industrial Crude Distillation Unit SOP</text:h>
      <text:p>The CDU-2 atmospheric tower operates at a target furnace outlet temperature of 365 degrees Celsius.</text:p>
      <text:p>Maximum permissible coil outlet temperature is strictly capped at 370 degrees Celsius.</text:p>
      <text:p>Desalter operating pressure is maintained at 8.2 barg with 5.5% wash water injection rate.</text:p>
    </office:text>
  </office:body>
</office:document-content>'''
    zf.writestr("content.xml", content_xml)

# 2. Create a sample XLSX spreadsheet
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "DischargeLimits"
ws.append(["Parameter", "MaxPermissibleLimit", "Unit"])
ws.append(["Biochemical Oxygen Demand (BOD)", 30.0, "mg/L"])
ws.append(["Chemical Oxygen Demand (COD)", 125.0, "mg/L"])
ws.append(["Oil & Grease", 5.0, "mg/L"])
xlsx_buf = io.BytesIO()
wb.save(xlsx_buf)

# 3. Upload both documents via API
doc1_res = client.post("/api/documents", files={"file": ("Refinery_CDU_SOP.odt", odt_buf.getvalue(), "application/vnd.oasis.opendocument.text")})
doc2_res = client.post("/api/documents", files={"file": ("Plant_Effluent_Specs.xlsx", xlsx_buf.getvalue(), "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")})

print(f"Document 1 Indexed: {doc1_res.json()['filename']} (ID: {doc1_res.json()['id']})")
print(f"Document 2 Indexed: {doc2_res.json()['filename']} (ID: {doc2_res.json()['id']})")

## Step 5: Deterministic Spreadsheet Numerical Intelligence
Privora computes exact column totals, averages, and coordinate cell ranges directly from structured rows, eliminating LLM arithmetic hallucinations.

In [ ]:
doc2_id = doc2_res.json()['id']
analytics_res = client.get(f"/api/documents/{doc2_id}/tabular-analytics")
analytics_data = analytics_res.json()

print("Sheet Name:", analytics_data["sheets"][0]["sheet_name"])
for col in analytics_data["sheets"][0]["numeric_columns"]:
    print(f"Column: {col['column']} ({col['cell_range']})")
    print(f"  Sum: {col['total']}, Average: {col['mean']}, Min: {col['min']}, Max: {col['max']}, Count: {col['count']}")

## Step 6: Okapi BM25 Retrieval & Page-Aware Ranking
We submit technical queries against the indexed database and inspect the retrieved passage excerpts, scores, and source file citations.

In [ ]:
query = "What is the target furnace outlet temperature and desalter pressure in the CDU SOP?"
chat_res = client.post("/api/chat", json={"message": query, "top_k": 3})
chat_data = chat_res.json()

print(f"Query: {query}")
print(f"Support Level: {chat_data.get('support_level')}")
print(f"Answer Preview:\n{chat_data.get('answer')}\n")

print("Retrieved Source Citations:")
for i, cit in enumerate(chat_data.get("citations", [])):
    print(f"  [{i+1}] {cit['filename']} ({cit.get('section') or 'Body'}): {cit['excerpt'][:120]}...")

## Step 7: Unsupported Query Defense
When asked an out-of-scope query that is not backed by evidence in the indexed documents, Privora strictly refuses to hallucinate.

In [ ]:
unsupported_query = "What is the corporate cafeteria lunch menu and executive flight allowance?"
unsupp_res = client.post("/api/chat", json={"message": query, "top_k": 3})
unsupp_data = unsupp_res.json()

print(f"Query: {unsupported_query}")
print(f"Support Level: {unsupp_data.get('support_level')}")
print(f"Citations Count: {len(unsupp_data.get('citations', []))}")
print(f"Answer: {unsupp_data.get('answer')}")

## Step 8: Empirical Benchmark Evaluation Suite
Privora contains a built-in empirical evaluation suite with ground truth across 6 categories.

In [ ]:
bm_res = client.get("/api/benchmark")
bm_data = bm_res.json()

print("Benchmark Suite:", bm_data["suite_name"])
print("Summary Metrics:")
for k, v in bm_data["summary_metrics"].items():
    print(f"  - {k}: {v}")

---

## ⚙️ Google Colab Limitations vs Native Windows Desktop Edition

| Dimension | Google Colab Preview | Windows Desktop Edition (Production) |
| :--- | :--- | :--- |
| **Operating System** | Linux (Ubuntu cloud VM) | Windows 10 / Windows 11 (64-bit) |
| **User Interface** | Headless Python API & Jupyter Cells | Native Standalone Desktop (Edge App Mode, dark GUI) |
| **Installation** | Transient cloud session | Zero-privilege installer (`Privora-Setup-1.1.0.exe`) |
| **Air-Gap Security** | Cloud VM (demo/evaluation only) | Strict on-premise air-gap with zero network leakage |
| **VRAM & Persistence**| Ephemeral 12-hour session | Permanent SQLite storage in `%LOCALAPPDATA%\Privora` |
| **Visual QA (Moondream)**| Limited by free GPU allocation | Full GPU-accelerated VLM and ONNX OCR |